# MedSegDiff-COCO — train on Kaggle (2× T4)

Joint **colorization + semantic segmentation** (grayscale → ab colors + 182-class map).

1. **Settings**: Accelerator **GPU T4 ×2**, Internet **On**, add the dataset `dntai2/cocostuf-2017`.
2. Fill in cell 1 (`WANDB_API_KEY`, `RESUME`), then *Run All* to try it, or **Save Version → Save & Run All (Commit)** to train in the background (≤ 12h; the script stops and saves a checkpoint at `MAX_HOURS`).
3. When done: download `/kaggle/working/run/checkpoints/latest.pt` (Output tab) and upload it to Google Drive → next session put that link into `RESUME`.

The checkpoint contains model + EMA + optimizer + scaler + step/epoch + all args + metrics + wandb run id ⇒ training continues exactly where it stopped (same wandb run).

In [ ]:
# 1. Configuration
WANDB_API_KEY = ""          # paste your wandb key temporarily (or create the Kaggle secret WANDB_API_KEY); empty = wandb off
RESUME = ""                 # "" = new run / continue in /kaggle/working/run ; or a Google Drive link / URL / path of a checkpoint
INIT_FROM = ""              # weights only (fine-tune): new optimizer, step = 0
RUN_NAME = "coco164k-256"

DATA_DIR = "/kaggle/input/datasets/dntai2/cocostuf-2017"
OUT_DIR = "/kaggle/working/run"
MAX_HOURS = 11.3            # stop + checkpoint before the 12h session limit
EXTRA_ARGS = [
    "--image_size", "256", "--batch_size", "3", "--grad_accum", "2",   # effective batch 3*2*2 = 12
    "--use_checkpoint", "False",          # fastest on T4 (8.1 img/s, 13.2 GB); set True + batch 4 if OOM
    "--lr", "1e-4", "--hw_lr_mult", "5", "--warmup_steps", "1000",
    "--save_minutes", "30", "--save_interval", "5000", "--keep_ckpts", "1",
    "--vis_interval", "1000", "--vis_n", "4", "--log_interval", "50",
]

In [ ]:
# 2. Code + dependencies
import os, subprocess, sys
REPO = "https://github.com/matbinhtinh/medsegdiff-coco.git"
CODE = "/kaggle/working/medsegdiff-coco"
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", CODE, "reset", "-q", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
subprocess.run("pip install -q scikit-image blobfile gdown wandb", shell=True)
print(subprocess.run(["git", "-C", CODE, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. wandb (optional)
key = WANDB_API_KEY.strip()
if not key:
    try:
        from kaggle_secrets import UserSecretsClient
        key = (UserSecretsClient().get_secret("WANDB_API_KEY") or "").strip()
    except Exception:
        key = ""
env = dict(os.environ)
if key:
    env["WANDB_API_KEY"] = key
    print("wandb: ON")
else:
    print("wandb: OFF")

In [ ]:
# 4. Train (DDP on all GPUs). Output is streamed; the log is also in OUT_DIR/log.txt
import torch
ngpu = max(torch.cuda.device_count(), 1)
cmd = ["torchrun", "--standalone", f"--nproc_per_node={ngpu}", f"{CODE}/scripts/coco_train.py",
       "--data_dir", DATA_DIR, "--out_dir", OUT_DIR, "--max_hours", str(MAX_HOURS),
       "--resume", RESUME or "auto", "--wandb", "True" if key else "False", "--wandb_name", RUN_NAME] + EXTRA_ARGS
if INIT_FROM:
    cmd += ["--init_from", INIT_FROM]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit code:", proc.wait())

In [ ]:
# 5. Results: checkpoints + latest sample images
import glob
from IPython.display import Image as IPImage, display
!ls -la {OUT_DIR}/checkpoints
imgs = sorted(glob.glob(f"{OUT_DIR}/samples/*.jpg"))
if imgs:
    print(imgs[-1]); display(IPImage(imgs[-1]))

In [ ]:
# 6. (optional) small weights-only file (EMA) for inference / sharing
sys.path.insert(0, CODE)
from guided_diffusion import coco_ckpt
src = coco_ckpt.find_latest(f"{OUT_DIR}/checkpoints")
if src:
    print(coco_ckpt.export_weights(src, f"{OUT_DIR}/ema_weights.pt"))